# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

##Install dependencies

In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma \
    chromadb sentence-transformers pypdf tiktoken requests tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204

##Download the PDF (Crime and Punishment)

In [2]:
import requests
from pathlib import Path

PDF_URL = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
PDF_PATH = Path("crime_and_punishment.pdf")

if not PDF_PATH.exists():
    resp = requests.get(PDF_URL, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)
    print(f"Downloaded {len(resp.content)/1024:.1f} KB")
else:
    print("Already downloaded")

Downloaded 2430.5 KB


##Extract text from PDF

In [3]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
print(f"Total pages: {len(reader.pages)}")

raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"

print(f"Total characters extracted: {len(raw_text):,}")
print(raw_text[2000:2500])

Total pages: 767
Total characters extracted: 1,172,469
 a few minutes of life before me. 
I thought of you and your dear ones and I contrived to kiss 
Plestcheiev and Dourov, who were next to me, and to bid 
them farewell. Suddenly the troops beat a tattoo, we were 
unbound, brought back upon the scaffold, and informed 
that his Majesty had spared us our lives.’ The sentence was 
commuted to hard labour.
One of the prisoners, Grigoryev, went mad as soon as he 
was untied, and never regained his sanity.
The intense suffering of this experience left a


##Clean/normalize extracted text

In [4]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"\r\n", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # drop obvious page-number-only lines
    text = re.sub(r"\n\d{1,4}\n", "\n", text)
    return text.strip()

cleaned_text = clean_text(raw_text)
print(f"Cleaned length: {len(cleaned_text):,} chars")

Cleaned length: 1,172,428 chars


##Chunking (3 strategies: small/medium/large)

In [5]:
!pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_CONFIGS = {
    "small":  {"chunk_size": 300,  "chunk_overlap": 50},
    "medium": {"chunk_size": 800,  "chunk_overlap": 150},
    "large":  {"chunk_size": 1500, "chunk_overlap": 300},
}

chunks_by_strategy = {}

for name, cfg in CHUNK_CONFIGS.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    docs = splitter.create_documents([cleaned_text])
    for i, d in enumerate(docs):
        d.metadata = {"strategy": name, "chunk_id": i}
    chunks_by_strategy[name] = docs
    print(f"{name:>7}: {len(docs):5d} chunks  (avg {sum(len(d.page_content) for d in docs)//len(docs)} chars)")

  small:  4390 chunks  (avg 270 chars)
 medium:  1804 chunks  (avg 769 chars)
  large:   978 chunks  (avg 1469 chars)


##Load HuggingFace embedding models (MiniLM, BGE)

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge":    "BAAI/bge-small-en-v1.5",
}

embedders = {
    name: HuggingFaceEmbeddings(model_name=model_id, encode_kwargs={"normalize_embeddings": True})
    for name, model_id in EMBEDDING_MODELS.items()
}
print("Embedding models ready:", list(embedders.keys()))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding models ready: ['minilm', 'bge']


##Index all chunk×embedding combinations into ChromaDB

In [7]:
from langchain_chroma import Chroma
import itertools, time

CHROMA_DIR = "chroma_store"
vectorstores = {}

for strat_name, embed_name in itertools.product(chunks_by_strategy, embedders):
    key = f"{strat_name}__{embed_name}"
    t0 = time.time()
    vs = Chroma.from_documents(
        documents=chunks_by_strategy[strat_name],
        embedding=embedders[embed_name],
        collection_name=key,
        persist_directory=CHROMA_DIR,
    )
    vectorstores[key] = vs
    print(f"Indexed {key:<25} in {time.time()-t0:6.1f}s  ({len(chunks_by_strategy[strat_name])} chunks)")

Indexed small__minilm             in    9.3s  (4390 chunks)
Indexed small__bge                in   11.1s  (4390 chunks)
Indexed medium__minilm            in    6.5s  (1804 chunks)
Indexed medium__bge               in   10.1s  (1804 chunks)
Indexed large__minilm             in    5.1s  (978 chunks)
Indexed large__bge                in   10.1s  (978 chunks)


##Define run_query() + sanity-check on 2 test queries

In [9]:
def run_query(vs, query, k=3):
    results = vs.similarity_search_with_relevance_scores(query, k=k)
    return results

TEST_QUERIES = [
    "Raskolnikov kills the old pawnbroker with an axe",
    "Sonia tells Raskolnikov to confess and accept suffering",
    "Raskolnikov describes his theory that extraordinary men are above the law",
    "Porfiry Petrovich questions and suspects Raskolnikov of the murder",
    "Raskolnikov confesses his crime to Sonia",
]

for q in TEST_QUERIES[:2]:
    print("="*100)
    print("QUERY:", q)
    for doc, score in run_query(vectorstores["medium__bge"], q):
        print(f"  score={score:.3f}  |  {doc.page_content[:180].strip()}...")

QUERY: Raskolnikov kills the old pawnbroker with an axe
  score=0.643  |  and waited. Water was brought.
‘It was I …’ began Raskolnikov.
‘Drink some water.’
Raskolnikov refused the water with his hand, and softly 
and brokenly, but distinctly said:
‘It w...
  score=0.579  |  which gave one tinkle, then gently, as though reflecting and 
looking about him, began touching the door-handle pull -
ing it and letting it go to make sure once more that it was...
  score=0.542  |  and quickly back into the flat and closing the door behind 
him. Then he took the hook and softly, noiselessly, fixed it 
in the catch. Instinct helped him. When he had done this,...
QUERY: Sonia tells Raskolnikov to confess and accept suffering
  score=0.664  |  hesitation and suffering, he quickly opened the door and 
looked at Sonia from the doorway. She was sitting with her 
Crime and Punishment0
elbows on the table and her face in he...
  score=0.643  |  am most ready, most ready to show compassion, if pover -
t

##Full benchmark loop across all queries × all combos

In [10]:
import pandas as pd

rows = []
for key, vs in vectorstores.items():
    strat, embed_name = key.split("__")
    for q in TEST_QUERIES:
        results = run_query(vs, q, k=3)
        avg_score = sum(s for _, s in results) / len(results)
        top_score = max(s for _, s in results)
        rows.append({
            "strategy": strat,
            "embedding": embed_name,
            "query": q,
            "avg_top3_score": avg_score,
            "top1_score": top_score,
        })

df = pd.DataFrame(rows)
summary = df.groupby(["strategy", "embedding"])[["avg_top3_score", "top1_score"]].mean().sort_values("avg_top3_score", ascending=False)
summary

avg_top3_score  top1_score
strategy embedding                            
medium   bge              0.648897    0.676294
large    bge              0.636099    0.655889
small    bge              0.632331    0.661790
         minilm           0.556580    0.591109
medium   minilm           0.533755    0.571614
large    minilm           0.499762    0.541878

## Findings

- **Query phrasing significantly affects retrieval quality.** Initial testing used abstract, thematic queries (e.g. "What does Sonia represent to Raskolnikov?"), which returned only loosely related passages. Switching to concrete, narrative-style queries that mirror how the book actually describes events (e.g. "Raskolnikov kills the old pawnbroker with an axe") produced noticeably more relevant top matches — for example, this query correctly retrieved the actual break-in and murder scene as its top-3 results. This highlights that embedding-based retrieval on literary text performs best when queries resemble concrete scene descriptions rather than abstract interpretive questions.

- **Embedding model matters more than chunk size.** Every `bge` configuration outperforms every `minilm` configuration at every chunk size. `BAAI/bge-small-en-v1.5` is the clear choice for this corpus.

- **Best overall configuration: `medium` chunking (800 chars / 150 overlap) + `bge-small-en-v1.5`**, with an average top-3 relevance score of 0.649 and a top-1 score of 0.676 — the highest of all six configurations tested. `large/bge` (0.636) and `small/bge` (0.632) trail close behind, while all `minilm` configurations score well below 0.56.

- **For BGE, medium chunks now edge out large chunks** — likely because medium-sized chunks (~800 chars) balance enough context to capture full sentences/actions without diluting relevance across unrelated content, which can happen with 1500-char chunks.

- **For MiniLM, smaller chunks still perform better** (small 0.557 > medium 0.534 > large 0.500), consistent with earlier testing — MiniLM's embeddings appear to degrade on longer, more diffuse text.

- **Recommended configuration for this corpus**: `medium` chunking + `bge-small-en-v1.5` embeddings, indexed with cosine similarity. This combination retrieves the most relevant passages while keeping chunk count (and thus indexing time/storage) lower than the `large` alternative.

| strategy | embedding | avg_top3_score | top1_score |
|---|---|---|---|
| medium | bge | 0.649 | 0.676 |
| large | bge | 0.636 | 0.656 |
| small | bge | 0.632 | 0.662 |
| small | minilm | 0.557 | 0.591 |
| medium | minilm | 0.534 | 0.572 |
| large | minilm | 0.500 | 0.542 |